## TF-IDF Word + Character n-grams + Logistic Regression



In [2]:

import pandas as pd
import numpy as np
import os
import joblib

# Divisão dos dados
from sklearn.model_selection import train_test_split

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

# GridSearch
from sklearn.model_selection import GridSearchCV

# Métricas
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Sentence Transformer
from sentence_transformers import SentenceTransformer

from sklearn.preprocessing import StandardScaler

### 1. Carregamento dos dados de treino

In [31]:
# df_train = pd.read_csv("../data/train.csv")
# df_train = pd.read_csv("../data/train_noDuplicates.csv")
df_train = pd.read_csv("../data/train_noSameClassDuplicates.csv")

# print(df_train.shape)
# print(df_train.columns)
# print(df_train.head())
# print(df_train.isnull().sum())

### 2. Definição das colunas de texto e categoria

In [32]:
X = df_train["resp_text"]
y = df_train["clarity"]

# print("X:", X.shape)
# print("y:", y.shape)
# print("\nDistribuição das classes:")
# print(y.value_counts())
# print("\nProporção das classes:")
# print(y.value_counts(normalize=True))

### 3. Divisão do conjunto em treino e validação

In [33]:
# Separamos  para garantir que o modelo de representação não tenha acesso
# aos dados que vamos usar para avaliar o classificador

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y, # Preserva a distribuição das classes na divisão
    random_state=100
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

print("\nClasses no treino:")
print(y_train.value_counts())

print("\nClasses no teste:")
print(y_test.value_counts())

Treino: (15050,)
Teste: (3763,)

Classes no treino:
clarity
c5      5278
c234    5170
c1      4602
Name: count, dtype: int64

Classes no teste:
clarity
c5      1320
c234    1292
c1      1151
Name: count, dtype: int64


In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 1. TF-IDF de palavras
# ==========================================

tfidf_word = TfidfVectorizer(
    ngram_range=(1, 2),   # unigramas + bigramas
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_word = tfidf_word.fit_transform(X_train)
X_test_word = tfidf_word.transform(X_test)


# ==========================================
# 2. TF-IDF de caracteres
# ==========================================

tfidf_char = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),
    min_df=2,
    sublinear_tf=True
)

X_train_char = tfidf_char.fit_transform(X_train)
X_test_char = tfidf_char.transform(X_test)


# ==========================================
# 3. Combinar as duas representações
# ==========================================

X_train_combined = hstack([
    X_train_word,
    X_train_char
])

X_test_combined = hstack([
    X_test_word,
    X_test_char
])


# ==========================================
# 4. Logistic Regression
# ==========================================

model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    class_weight='balanced'
)

model.fit(X_train_combined, y_train)


# ==========================================
# 5. Predição
# ==========================================

y_pred = model.predict(X_test_combined)


# ==========================================
# 6. Avaliação
# ==========================================

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro-F1: {macro_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.4372
Macro-F1: 0.4359

Classification Report:
              precision    recall  f1-score   support

          c1       0.42      0.44      0.43      1151
        c234       0.39      0.37      0.38      1292
          c5       0.49      0.50      0.50      1320

    accuracy                           0.44      3763
   macro avg       0.44      0.44      0.44      3763
weighted avg       0.44      0.44      0.44      3763



### 7. Melhorando o pré-processamento de texto

A acurácia de 0.46 sugere que podemos melhorar a representação do texto. A limpeza e normalização do texto são cruciais para que o modelo foque nas informações mais relevantes. Podemos:

*   **Remover Stopwords:** Palavras muito comuns (como 'o', 'a', 'de', 'para') que geralmente não adicionam muito significado à classificação.
*   **Remover Pontuação e Números:** Caracteres que podem ser irrelevantes para o significado do texto.
*   **Stemming/Lemmatização:** Reduzir as palavras às suas formas radicais ou bases para tratar variações de uma mesma palavra como um único token (ex: 'correndo', 'correr', 'corre' -> 'corr').

Vamos implementar uma função de limpeza de texto para aplicar essas transformações e, em seguida, reaplicar o TF-IDF e treinar o modelo.

In [7]:
# import re
# import nltk
# from nltk.corpus import stopwords
# from nltk.stem import RSLPStemmer # Stemmer para português

# # Baixar recursos necessários do NLTK (se ainda não tiverem sido baixados)
# try:
#     stopwords.words('portuguese')
# except LookupError:
#     nltk.download('stopwords')
# try:
#     RSLPStemmer()
# except LookupError:
#     nltk.download('rslp')

# stop_words = set(stopwords.words('portuguese'))
# stemmer = RSLPStemmer()

# def clean_text_for_words(text):
#     text = str(text) # Ensure text is string
#     # Remover URLs
#     text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
#     # Remover menções de usuários e hashtags (se relevantes para o contexto)
#     text = re.sub(r'@\w+|#\w+', '', text)
#     # Remover pontuação
#     text = re.sub(r'[^À-ſ\w\s]', ' ', text)  # Mantém letras acentuadas, alfanuméricos e espaços
#     # Remover números
#     text = re.sub(r'\d+', '', text)
#     # Converter para minúsculas
#     text = text.lower()
#     # Tokenização e remoção de stopwords
#     words = text.split()
#     words = [word for word in words if word not in stop_words]
#     # Stemming (redução para o radical da palavra) - Mantido comentado por enquanto
#     # words = [stemmer.stem(word) for word in words]
#     return ' '.join(words)

# def clean_text_for_chars(text):
#     text = str(text) # Ensure text is string
#     # Apenas converter para minúsculas para n-grams de caracteres
#     return text.lower()

# # Aplicar as funções de limpeza aos dados
# X_train_word_prep = X_train.apply(clean_text_for_words)
# X_test_word_prep = X_test.apply(clean_text_for_words)

# X_train_char_prep = X_train.apply(clean_text_for_chars)
# X_test_char_prep = X_test.apply(clean_text_for_chars)


# print("Original X_train sample:")
# print(X_train.iloc[0])
# print("\nCleaned X_train sample (for words, sem stemming):")
# print(X_train_word_prep.iloc[0])
# print("\nCleaned X_train sample (for chars, apenas lowercase):")
# print(X_train_char_prep.iloc[0])

### 8. Reaplicando TF-IDF e Retreinando o Modelo com Dados Limpos

In [8]:
# # ==========================================
# # 1. TF-IDF de palavras (com dados limpos especificos)
# # ==========================================

# # Reutilizar o tfidf_word, mas fit_transform com os dados limpos para palavras
# X_train_word_vec = tfidf_word.fit_transform(X_train_word_prep)
# X_test_word_vec = tfidf_word.transform(X_test_word_prep)


# # ==========================================
# # 2. TF-IDF de caracteres (com dados limpos especificos)
# # ==========================================

# # Reutilizar o tfidf_char, mas fit_transform com os dados limpos para caracteres
# X_train_char_vec = tfidf_char.fit_transform(X_train_char_prep)
# X_test_char_vec = tfidf_char.transform(X_test_char_prep)


# # ==========================================
# # 3. Combinar as duas representações (limpas)
# # ==========================================

# X_train_combined_cleaned = hstack([
#     X_train_word_vec,
#     X_train_char_vec
# ])

# X_test_combined_cleaned = hstack([
#     X_test_word_vec,
#     X_test_char_vec
# ])

# print("Shape of X_train_combined_cleaned:", X_train_combined_cleaned.shape)
# print("Shape of X_test_combined_cleaned:", X_test_combined_cleaned.shape)


# # ==========================================
# # 4. Logistic Regression (com dados limpos)
# # ==========================================

# model_cleaned = LogisticRegression(
#     max_iter=1000,
#     C=1.0,
#     class_weight='balanced'
# )

# model_cleaned.fit(X_train_combined_cleaned, y_train)


# # ==========================================
# # 5. Predição (com dados limpos)
# # ==========================================

# y_pred_cleaned = model_cleaned.predict(X_test_combined_cleaned)


# # ==========================================
# # 6. Avaliação (com dados limpos)
# # ==========================================

# accuracy_cleaned = accuracy_score(y_test, y_pred_cleaned)

# print(f"\nAccuracy with cleaned data: {accuracy_cleaned:.4f}")
# print("\nClassification Report with cleaned data:")
# print(classification_report(y_test, y_pred_cleaned))

### 9. Comparando o peso das features (palavras vs. caracteres)

In [9]:
# import numpy as np

# # Obter os coeficientes do modelo
# coefficients = model_cleaned.coef_

# # Obter os nomes das features de palavras e caracteres
# word_features = tfidf_word.get_feature_names_out()
# char_features = tfidf_char.get_feature_names_out()

# num_word_features = len(word_features)
# num_char_features = len(char_features)

# # Separar os coeficientes para palavras e caracteres
# # A ordem é importante: primeiro palavras, depois caracteres
# word_coefs = coefficients[:, :num_word_features]
# char_coefs = coefficients[:, num_word_features:num_word_features + num_char_features]

# # Obter os nomes das classes
# class_names = model_cleaned.classes_

# print("\n--- Coeficientes das Features (Palavras vs. Caracteres) ---")

# for i, class_name in enumerate(class_names):
#     print(f"\nClasse: {class_name}")

#     # Top 10 palavras mais importantes para a classe
#     top_word_indices = np.argsort(word_coefs[i])[-10:][::-1]
#     print("  Top 10 Palavras Mais Positivas:")
#     for idx in top_word_indices:
#         print(f"    {word_features[idx]}: {word_coefs[i, idx]:.4f}")

#     # Top 10 palavras mais importantes contra a classe (mais negativas)
#     bottom_word_indices = np.argsort(word_coefs[i])[:10]
#     print("  Top 10 Palavras Mais Negativas:")
#     for idx in bottom_word_indices:
#         print(f"    {word_features[idx]}: {word_coefs[i, idx]:.4f}")

#     # Top 10 caracteres mais importantes para a classe
#     top_char_indices = np.argsort(char_coefs[i])[-10:][::-1]
#     print("  Top 10 Caracteres (n-grams) Mais Positivos:")
#     for idx in top_char_indices:
#         print(f"    {char_features[idx]}: {char_coefs[i, idx]:.4f}")

#     # Top 10 caracteres mais importantes contra a classe (mais negativas)
#     bottom_char_indices = np.argsort(char_coefs[i])[:10]
#     print("  Top 10 Caracteres (n-grams) Mais Negativos:")
#     for idx in bottom_char_indices:
#         print(f"    {char_features[idx]}: {char_coefs[i, idx]:.4f}")

### **GridSearch - Regressão Logística**




In [11]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression

pipeline_lr = Pipeline([
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

param_grid_lr = {
    "clf__C": [0.01, 0.1, 1, 10, 100]
}

In [12]:
cv_grid = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_lr = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=param_grid_lr,
    scoring="accuracy",
    cv=cv_grid,
    n_jobs=-1,
    return_train_score=True,
    verbose=3
)

grid_lr.fit(X_train_combined, y_train)

print("Melhores parâmetros:")
print(grid_lr.best_params_)

print(f"Melhor CV Accuracy: {grid_lr.best_score_:.4f}")

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Melhores parâmetros:
{'clf__C': 0.1}
Melhor CV Accuracy: 0.4620


### **Validação Cruzada - variando seeds**

In [35]:
pipeline_lr_best = Pipeline([
    ("clf", LogisticRegression(
        C=0.1, # Atualizar aqui de acordo com GridSearch
        max_iter=1000,
        class_weight="balanced"
    ))
])

In [16]:
seeds = [42, 100, 123]

for seed in seeds:

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )

    results = cross_validate(
        pipeline_lr_best,
        X_train_combined,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "f1_macro": "f1_macro"
        },
        return_train_score=True,
        n_jobs=-1
    )

    print(f"\n===== SEED {seed} =====")

    print(
        f"Accuracy: "
        f"{results['test_accuracy'].mean():.4f} "
        f"+/- {results['test_accuracy'].std():.4f}"
    )

    print(
        f"Macro-F1: "
        f"{results['test_f1_macro'].mean():.4f} "
        f"+/- {results['test_f1_macro'].std():.4f}"
    )

    print(
        f"Train Accuracy: "
        f"{results['train_accuracy'].mean():.4f}"
    )


===== SEED 42 =====
Accuracy: 0.4620 +/- 0.0106
Macro-F1: 0.4573 +/- 0.0102
Train Accuracy: 0.6167

===== SEED 100 =====
Accuracy: 0.4604 +/- 0.0087
Macro-F1: 0.4557 +/- 0.0084
Train Accuracy: 0.6153

===== SEED 123 =====
Accuracy: 0.4569 +/- 0.0107
Macro-F1: 0.4518 +/- 0.0104
Train Accuracy: 0.6177


In [36]:
pipeline_lr_best.fit(
    X_train_combined,
    y_train
)

y_pred = pipeline_lr_best.predict(X_test_combined)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(
    f"Macro-F1: "
    f"{f1_score(y_test, y_pred, average='macro'):.4f}"
)

Accuracy: 0.4451
Macro-F1: 0.4402
